## Clustering: Funciones

Constantes y funciones compartidas

### Imports

In [ ]:
import json, tracemalloc
import numpy as np
from sklearn.metrics import adjusted_rand_score, silhouette_score

### Constantes

- `SEED` = 42: misma semilla en muestras y modelos

- `SUJETOS`: los dos tipos de actor que se clusterizan y su llave

- `datos_dir`: datos que pasan de un notebook al siguiente

- `N_SIL` = 3,000: puntos para estimar la silueta

In [ ]:
SEED = 42
SUJETOS = {"entidad": "entidad_key", "proveedor": "proveedor_key"}
datos_dir = Path.cwd() / "temp" / "datos"
datos_dir.mkdir(parents=True, exist_ok=True)
N_SIL = 3000

### Parámetros elegidos

| Parámetro | Valor | Por qué | Sección |
|---|---|---|---|
| `MIN_PROCESOS` | entidad 5, proveedor 3 | Los vectores de proveedores dejan de repetirse; porcentajes son informativos | `a_preparacion` |
| `DESCARTE_FIJO` | `compite`, `es_consorcio`, `pct_servicios` | Constante, casi constante tras el filtro y combinación lineal | `a_preparacion` |
| `UMBRAL_CORR` | 0.85 | Quita lo redundante sin borrar volumen ni gasto | `a_preparacion` |
| `VAR_PCA` | 0.85 | 6 componentes; la silueta casi no cambia frente al espacio completo | `a_preparacion`, `b_kmeans` |
| `K` | 4 y 4 | Coinciden el codo y la silueta máxima | `b_kmeans` |
| `RUIDO_MAX` | 0.10 | La configuración de DBSCAN es la misma con 5% y 10% | `c_densidad` |
| `RUIDO_MAX_H` | 0.30 | Menor tope con una configuración de HDBSCAN válida en ambos sujetos | `c_densidad` |
| `N_JER` | 5,000 | Mejor silueta con ~0.1 GB por árbol; todos los proveedores serían ~0.7 GB | `d_jerarquico` |
| `ENLACE` | ward | Único con grupos balanceados; average y single encadenan | `d_jerarquico` |
| `K_JER` | 4 y 4 | Máximo de silueta en proveedores y comparable con KMeans | `d_jerarquico` |
| `UMBRAL_HHI` | 0.25 | Umbral usual de alta concentración, el mismo del EDA | `e_analisis` |
| `CONSENSO` | 2 | Con 1, el ruido de HDBSCAN infla la lista; con 3, quedan solo los extremos | `e_analisis` |

In [ ]:
MIN_PROCESOS = {"entidad": 5, "proveedor": 3}
DESCARTE_FIJO = ["compite", "es_consorcio", "pct_servicios"]
UMBRAL_CORR = 0.85
VAR_PCA = 0.85
K = {"entidad": 4, "proveedor": 4}
RUIDO_MAX = 0.10
RUIDO_MAX_H = 0.30
N_JER = 5000
ENLACE = "ward"
K_JER = {"entidad": 4, "proveedor": 4}
UMBRAL_HHI = 0.25
CONSENSO = 2

### Configuración entre notebooks

`config.json` guarda lo que cada notebook decide (features, componentes, k…)

In [ ]:
def leer_config():
    ruta = datos_dir / "config.json"
    return json.loads(ruta.read_text()) if ruta.exists() else {}

def guardar_config(config):
    (datos_dir / "config.json").write_text(json.dumps(config, indent=1, ensure_ascii=False))

### Datos locales

`cargar_local`: lee `prep_<sujeto>` con pandas y devuelve la tabla y la matriz X de los primeros `n_comp` componentes

In [ ]:
def cargar_local(s, n_comp):
    d = pd.read_parquet(datos_dir / f"prep_{s}")
    X = np.vstack(d["pcs"].to_numpy())[:, :n_comp]
    return d, X

def muestra_estratificada(d, n, por):
    # misma proporción de cada grupo (p. ej. clúster de KMeans) que en el total
    if n >= len(d):
        return d.index.to_numpy()
    frac = n / len(d)
    return (d.groupby(por, group_keys=False).apply(lambda g: g.sample(max(1, round(len(g) * frac)), random_state=SEED))
             .index.to_numpy())

### Métricas

- `silueta`: silueta muestreada, ignorando el ruido (-1) de los métodos de densidad

- `ari`: coincidencia entre dos particiones (1 = iguales, 0 = al azar)

- `medir`: tiempo y memoria pico (`tracemalloc`) de una función

In [ ]:
def silueta(X, lab, n=N_SIL):
    lab = np.asarray(lab); m = lab >= 0
    if len(set(lab[m])) < 2 or m.sum() < 10:
        return np.nan
    return float(silhouette_score(X[m], lab[m], sample_size=int(min(n, m.sum())), random_state=SEED))

def ari(a, b):
    a, b = np.asarray(a), np.asarray(b); m = (a >= 0) & (b >= 0)
    return float(adjusted_rand_score(a[m], b[m]))

def medir(f, *args, **kw):
    tracemalloc.start(); t0 = time.time()
    r = f(*args, **kw)
    seg = time.time() - t0; pico = tracemalloc.get_traced_memory()[1] / 1024**2
    tracemalloc.stop()
    return r, round(seg, 2), round(pico, 1)

### Perfiles

`rasgos`: describe un clúster o un actor por sus features más alejadas de la media, en desviaciones estándar (σ)

In [ ]:
def rasgos(z, cols, top=3):
    z = pd.Series(z, index=cols).astype(float)
    orden = z.abs().sort_values(ascending=False).index[:top]
    return ", ".join(f"{c} {'↑' if z[c] > 0 else '↓'}{abs(z[c]):.1f}σ" for c in orden)

def perfil_z(d, feats, col):
    # media de cada feature estandarizada por grupo, con sus rasgos principales
    Z = pd.DataFrame(np.vstack(d["z"].to_numpy()), columns=feats, index=d.index)
    p = Z.groupby(d[col]).mean()
    p["rasgos"] = [rasgos(r[feats].to_numpy(), feats) for _, r in p.iterrows()]
    return p.round(2)